In [1]:
# 1. IMPORT LIBRARIES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
# 2. LOAD DATASET

df = pd.read_csv(
    "IMDB_Dataset.csv",
    encoding_errors='ignore',
    on_bad_lines='skip',
    engine='python'
)

print (df)

# Drop missing values
df = df.dropna()

print("\nUnique sentiment labels (raw):")
print(df['sentiment'].unique())

print("Nulls in sentiment:", df['sentiment'].isnull().sum())

print("\nNulls per column after dropna:")
print(df.isnull().sum())

                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49608  I thought this movie did a down right good job...  positive
49609  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49610  I am a Catholic taught in parochial elementary...  negative
49611  I'm going to have to disagree with the previou...  negative
49612  No one expects the Star Trek movies to be high...  negative

[49613 rows x 2 columns]

Unique sentiment labels (raw):
['positive' 'negative']
Nulls in sentiment: 0

Nulls per column after dropna:
review       0
sentiment    0
dtype: int64


In [3]:
# 3. FIX COLUMN NAMES
df.columns = ['review', 'sentiment']

In [ ]:
# 4. CLEAN TEXT

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'<.*?>','', text)       # remove HTML tags
    text = re.sub(r'[a-zA-Z ]', '', text)  # keep only letters and spaces
    return text

df['review'] = df['review'].apply(clean_text)

In [ ]:
# 5. CONVERT LABELS

df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print("\nSentiment Count (0=negative, 1=positive):")
print(df['sentiment'].value_counts())

In [ ]:
# 6. TF-IDF VECTORIZATION

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['review']).toarray()
y = df['sentiment']

In [ ]:
# 7. TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# 8. BUILD MODEL

model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))   # binary output

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# 9. TRAIN MODEL

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

In [ ]:
# 10. PLOT ACCURACY GRAPH

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title("Accuracy Graph")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
# 11. FINAL ACCURACY

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("\nFinal Test Accuracy:", accuracy)